In [11]:
class BPETokenizer():
    def __init__(self):
        self.merge = {}
        self.id_2_char = {}
        self.char_2_id = {}
    def train(self, input_texts, vacab_size):
        '''
        BPE算法的训练过程
        :param input_texts: 输入语料库
        :param vocal_size: 目标构建的词典的大小
        :return: 
        '''
        
        #1.对输入语料进行切分
        unique_chrs = list(set(list(input_texts)))
        #2.得到一个初始化的字典
        id_2_char = {idx:char for idx, char in enumerate(unique_chrs)}
        char_2_id = {char:idx for idx, char in enumerate(unique_chrs)}
        #3.利用字典对输入语料进行id化
        ids = [char_2_id[c] for c in input_texts]
        
        merge_times = vacab_size-len(unique_chrs)
        vacab_size = len(unique_chrs)-1
        merge = {}
        #4.训练，合并子词，直到字典的大小达到val_size
        for i in range(merge_times):
            if len(ids) == 1:
                break
            # 统计相邻子词出现的频率
            stats = self.stats(ids)
            
            #找出出现频率最高的相邻子词对
            pair = max(stats, key=stats.get)
            vacab_size += 1
            id_2_char[vacab_size] = id_2_char[pair[0]] + id_2_char[pair[1]]
            char_2_id[id_2_char[pair[0]] + id_2_char[pair[1]]] = vacab_size
            merge[pair] = vacab_size
            
            #根据当前的词典，合并ids
            ids = self.merge_ids(ids, pair, vacab_size)
            
        self.merge=merge
        self.char_2_id = char_2_id
        self.id_2_char = id_2_char
            
    def stats(self, ids):
        '''
        统计相邻子词出现的频率
        :param ids: 根据当前词典索引化后的语料库
        :return: 
        '''
        #[1,2,3,4,1,2]
        #[1,2,3,4,1]
        #[2,3,4,1,2]
        count = {}
        for item in zip(ids[:-1], ids[1:]):
            count[item] = count.get(item, 0) + 1
        return count   
    
    def merge_ids(self, ids, pair, idx):
        '''
        合并语料库里面的相邻子词对，并更新ids
        :param ids:语料库未更新前的ids
        :param pair: 当前待合并的相邻子词对
        :param idx: 当前合并的相邻子词对在词典里的id
        :return: 
        '''
        new_ids = []
        i=0
        #ids: [1,2,3,4,5] [3,4] 
        while i<len(ids):
            if ids[i]==pair[0] and i<len(ids)-1 and ids[i+1]==pair[1]:
                new_ids.append(idx)
                i+=2
            else:
                new_ids.append(ids[i])
                i+=1
        return new_ids
    
    def encode(self, text):
        '''
        将输入文本进行切分并索引化
        :param text: 
        :return: 
        '''
        #1.对输入文本进行简单切分
        ids = [self.char_2_id[c] for c in text]
        print(ids)
        #2 利用merge词典，进行多次合并，得到最终的输出
        while len(ids)>=2:
            stats = self.stats(ids)
            pair = min(stats, key=lambda p:self.merge.get(p, float('inf')))
            if pair not in self.merge:
                break
            ids = self.merge_ids(ids, pair, self.merge[pair])
        return ids
    
    def decode(self, ids):
        '''
        将索引列表转化为文本
        :param ids: 
        :return: 
        '''
        return "".join([self.id_2_char[index] for index in ids])

In [12]:
t1 = BPETokenizer()

In [16]:
train_text = """
    hello, this is a training text. The tokenizer will split the text into words and assign an id
    to each word. This is a fantastic world.
    """
t1.train(input_texts=train_text, vacab_size=48)
t1.id_2_char

{0: 'r',
 1: 'k',
 2: 'e',
 3: '\n',
 4: 's',
 5: 'c',
 6: 'T',
 7: '.',
 8: ',',
 9: 'n',
 10: 'd',
 11: 'x',
 12: ' ',
 13: 'i',
 14: 'p',
 15: 'l',
 16: 'a',
 17: 'o',
 18: 'w',
 19: 'g',
 20: 't',
 21: 'f',
 22: 'z',
 23: 'h',
 24: '  ',
 25: ' t',
 26: 's ',
 27: 'is ',
 28: ' w',
 29: '\n  ',
 30: '\n    ',
 31: 'he',
 32: 'in',
 33: ' wo',
 34: ' wor',
 35: 'an',
 36: 'll',
 37: 'his ',
 38: 'his is ',
 39: 'his is a',
 40: ' te',
 41: ' tex',
 42: ' text',
 43: '. ',
 44: '. T',
 45: 'to',
 46: ' word',
 47: 'as'}

In [17]:
t1.encode("hello world")

[23, 2, 15, 15, 17, 12, 18, 17, 0, 15, 10]


[31, 36, 17, 34, 15, 10]

In [19]:
t1.decode([31, 36, 17, 34, 15, 10])

'hello world'